# CLEANING PIPELINE

# IMPORT LIABRARIES

In [1]:
import pandas as pd

import numpy as np

from sqlalchemy import create_engine

from urllib.parse import quote_plus

# MYSQL CONNECTION

In [2]:
username = "root"
password =  quote_plus("SQL_Admin19@#")
host = "127.0.0.1"
database = "hr_analytics"
port="3306"

## CREATE ENGINE

In [3]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# LOAD STAGING TABLES

In [4]:
people_data = pd.read_sql(

    "SELECT * FROM stg_people_data",

    con=engine
)

employment_history = pd.read_sql(

    "SELECT * FROM stg_people_employment_history",

    con=engine
)

# PROFILING

In [5]:
people_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4138 entries, 0 to 4137
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   employee_id        4138 non-null   str   
 1   gender             4138 non-null   str   
 2   race               4138 non-null   str   
 3   birth_date         4138 non-null   object
 4   education          4040 non-null   str   
 5   location           4138 non-null   str   
 6   location_city      4138 non-null   str   
 7   marital_status     4138 non-null   str   
 8   employment_status  4138 non-null   str   
dtypes: object(1), str(8)
memory usage: 291.1+ KB


In [6]:
employment_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 4138 entries, 0 to 4137
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           4138 non-null   str    
 1   first_name            4138 non-null   str    
 2   last_name             4138 non-null   str    
 3   department            4138 non-null   str    
 4   sub_department        4137 non-null   str    
 5   first_level_manager   4137 non-null   str    
 6   second_level_manager  4030 non-null   str    
 7   third_level_manager   3678 non-null   str    
 8   fourth_level_manager  3093 non-null   str    
 9   job_level             4138 non-null   str    
 10  salary                4138 non-null   float64
 11  hire_date             4138 non-null   object 
 12  term_date             729 non-null    object 
 13  term_type             729 non-null    str    
 14  term_reason           729 non-null    str    
 15  active_status         4138 non-n

# NULL VALIDATION 

In [7]:
def null_validation(df, table_name):

    null_percent = (

        df.isnull()
        .mean()
        * 100
    ).round(2)

    results = pd.DataFrame({

        "column_name": null_percent.index,

        "null_percentage": null_percent.values,

        "table_name": table_name
    })

    return results

In [8]:
people_nulls = null_validation(

    people_data,

    "stg_people_data"
)

employment_nulls = null_validation(

    employment_history,

    "stg_people_employment_history"
)

In [9]:
people_nulls

,column_name,null_percentage,table_name
0,employee_id,0.00,stg_people_data
1,gender,0.00,stg_people_data
2,race,0.00,stg_people_data
3,birth_date,0.00,stg_people_data
4,education,2.37,stg_people_data
5,location,0.00,stg_people_data
6,location_city,0.00,stg_people_data
7,marital_status,0.00,stg_people_data
8,employment_status,0.00,stg_people_data


In [10]:
employment_nulls

,column_name,null_percentage,table_name
0,employee_id,0.00,stg_people_employment_history
1,first_name,0.00,stg_people_employment_history
2,last_name,0.00,stg_people_employment_history
3,department,0.00,stg_people_employment_history
4,sub_department,0.02,stg_people_employment_history
5,first_level_manager,0.02,stg_people_employment_history
6,second_level_manager,2.61,stg_people_employment_history
7,third_level_manager,11.12,stg_people_employment_history
8,fourth_level_manager,25.25,stg_people_employment_history
9,job_level,0.00,stg_people_employment_history


# EXPORT RESULTS

In [11]:
people_nulls.to_csv(

    "../../docs/people_null_validation.csv",

    index=False
)

# DUPLICATE VALIDATION

In [12]:
# EMPLOYEE DUPLICATES

duplicate_employees = (

    people_data[
        people_data.duplicated(subset=['employee_id'],keep=False)]
)
duplicate_employees

,employee_id,gender,race,birth_date,education,location,location_city,marital_status,employment_status


In [13]:
# COUNT DUPLICATES

duplicate_employees.shape[0]

0

# BUSINESS RULE VALIDATION

## RULE 1 — HIRE DATE BEFORE TERM DATE

In [14]:
invalid_dates = employment_history[

    (employment_history['term_date'].notnull())
    &
    (employment_history['hire_date']
        >
    employment_history['term_date']
    )
]

In [15]:
len(invalid_dates)

0

# NEGATIVE SALARY CHECK

In [16]:
negative_salary = employment_history[

    employment_history['salary'] < 0
]
len(negative_salary)

0

# FUTURE HIRE DATE CHECK

In [17]:
future_hires = employment_history[
    employment_history['hire_date']
    >
    pd.Timestamp("2022-12-31").date()
]
len(future_hires)

0

# CEO VALIDATION

In [18]:
ceo_rows = employment_history[

    employment_history['first_level_manager']
    .isnull()
]
len(ceo_rows)

1

# SELF-REPORTING CHECK

In [19]:
self_reporting = employment_history[

    employment_history['employee_id']
    ==
    employment_history['first_level_manager']
]
len(self_reporting)

0

# ORPHAN MANAGER CHECK

In [20]:
orphan_managers = employment_history[

    ~employment_history['first_level_manager']
    .isin(
        employment_history['employee_id']
    )

    &

    employment_history['first_level_manager']
    .notnull()
]
len(orphan_managers)

0

# DATE STANDARDIZATION

In [ ]:
people_data['birth_date'] = pd.to_datetime(
    people_data['birth_date'],
    errors = 'coerce'
)

In [ ]:
people_data['birth_date'].info()

In [21]:
date_cols = [
    'hire_date',
    'term_date'
]

for col in date_cols:
    employment_history[col] = pd.to_datetime(
        employment_history[col],
        errors='coerce'
    )

In [22]:
employment_history['hire_date'].info()

<class 'pandas.Series'>
RangeIndex: 4138 entries, 0 to 4137
Series name: hire_date
Non-Null Count  Dtype        
--------------  -----        
4138 non-null   datetime64[s]
dtypes: datetime64[s](1)
memory usage: 32.5 KB


In [23]:
employment_history['term_date'].info()

<class 'pandas.Series'>
RangeIndex: 4138 entries, 0 to 4137
Series name: term_date
Non-Null Count  Dtype        
--------------  -----        
729 non-null    datetime64[s]
dtypes: datetime64[s](1)
memory usage: 32.5 KB


# STANDARDIZATION PIPELINE

In [24]:
categorical_cols_pd = ['gender', 'race', 'education', 'location', 'location_city', 'marital_status', 'employment_status']

In [25]:
categorical_cols_eh = ['department', 'sub_department', 'job_level', 'term_reason']

In [26]:
def check_value_counts(df, columns):
    for col in columns:
        print(f"\n{'='*50}")
        print(f"COLUMN : {col}")
        print(df[col].value_counts(dropna = False))

In [27]:
check_value_counts(people_data, categorical_cols_pd)


COLUMN : gender
gender
Male      2514
Female    1619
Other        5
Name: count, dtype: int64

COLUMN : race
race
Caucasian           3093
Asian                419
African American     211
Hispanic             206
Native American      110
Other                 99
Name: count, dtype: int64

COLUMN : education
education
Bachelor's degree     3260
Master's degree        641
Associate's degree     139
NaN                     98
Name: count, dtype: int64

COLUMN : location
location
Remote     3300
On-site     838
Name: count, dtype: int64

COLUMN : location_city
location_city
San Francisco    506
Los Angeles      412
Dallas           226
Houston          193
Indianapolis     178
New York         177
Phoenix          176
San Diego        175
Denver           172
San Jose         171
Philadelphia     170
Boston           170
Fort Worth       169
Columbus         166
Charlotte        161
Washington DC    159
Austin           158
San Antonio      157
Jacksonville     154
Chicago          154
S

In [28]:
check_value_counts(employment_history, categorical_cols_eh)


COLUMN : department
department
Software               783
Sales                  635
Marketing              407
HR                     401
Operations             395
Finance                365
Administration         208
Legal                  202
Procurement            194
R&D                    187
Product Development    186
Customer Service       174
Executive                1
Name: count, dtype: int64

COLUMN : sub_department
sub_department
Technical Support        303
Software Development     276
QA                       266
Account Management       265
Business Development     221
Sales Operations         204
Project Management       200
Supply Chain             189
Logistics                146
Public Relations         144
Training                 140
Marketing Research       140
Financial Planning       134
Benefits                 133
Recruiting               128
Advertising              123
Auditing                 117
Accounting               114
Facilities                75


# STANDARDIZE TEXT

In [29]:
def clean_extra_spaces(df, columns):

    for col in columns:

        # Remove leading/trailing spaces
        # and multiple internal spaces
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
        )

    return df

In [30]:
clean_extra_spaces(people_data, categorical_cols_pd).head()

,employee_id,gender,race,birth_date,education,location,location_city,marital_status,employment_status
0,12104572130,Female,Caucasian,1967-01-25,Master's degree,On-site,Los Angeles,Married,Full Time
1,3381966,Female,Caucasian,1990-12-26,Bachelor's degree,Remote,Washington DC,Single,Full Time
2,12868764,Male,Caucasian,1982-05-30,Bachelor's degree,On-site,San Francisco,Single,Full Time
3,17445638,Male,Caucasian,1984-08-05,Bachelor's degree,Remote,Columbus,Single,Full Time
4,19611331,Male,African American,1990-10-04,Bachelor's degree,Remote,Washington DC,Single,Full Time


In [31]:
clean_extra_spaces(employment_history, categorical_cols_eh).head()

,employee_id,first_name,last_name,department,sub_department,first_level_manager,second_level_manager,third_level_manager,fourth_level_manager,job_level,salary,hire_date,term_date,term_type,term_reason,active_status
0,12104572130,Kip,O'Finan,Executive,NaN,NaN,NaN,NaN,NaN,CEO,2468287.0,2017-06-28,NaT,NaN,NaN,1
1,3381966,Kate,Maceur,Legal,Intellectual Property,3278383811,1490361837,2591261030,12104572130,Individual Contributor,112274.0,2017-12-18,NaT,NaN,NaN,1
2,12868764,Bard,Kenneford,Software,Software Development,7216768130,6279119392,6268712051,12104572130,Individual Contributor,101769.0,2012-03-28,NaT,NaN,NaN,1
3,17445638,Saw,Sogg,Marketing,Public Relations,6725256317,2884263685,1949400041,12104572130,Individual Contributor,82641.0,2017-06-29,2019-04-03,Voluntary,Found a better opportunity,0
4,19611331,Cullen,Stiell,HR,Benefits,6064345413,8377149542,12104572130,NaN,Team Lead,69487.0,2021-04-12,NaT,NaN,NaN,1


# NULL HANDLING

In [32]:
people_data['education'] = (
    people_data['education'].fillna('Unknown')
)

In [33]:
people_data['education'].value_counts(dropna=False)

education
Bachelor's degree     3260
Master's degree        641
Associate's degree     139
Unknown                 98
Name: count, dtype: int64

# ANOMALY DETECTION

In [34]:
# SALARY OUTLIERS

Q1 = employment_history['salary'].quantile(0.25)
Q3 = employment_history['salary'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

salary_outliers = employment_history[
    (employment_history['salary'] < lower)
    |
    (employment_history['salary'] > upper)
]

In [35]:
len(salary_outliers)

39

# Do NOT blindly remove outliers.

High salaries may be:
    
- executives
- directors
- senior leadership

In [36]:
check = employment_history[
        (employment_history['salary'] < lower)
    |
    (employment_history['salary'] > upper)
]
check['job_level'].value_counts(dropna=False)

job_level
Director     19
Manager      16
Team Lead     3
CEO           1
Name: count, dtype: int64

# VERIFIED THEY ARE NOT OUTLIERS

THEY ARE:

Director     
Manager      
Team Lead     
CEO 

# REFERENTIAL INTEGRITY VALIDATION

Employment records must map to valid employees

In [37]:
# EMPLOYEE COVERAGE

missing_employees = employment_history[
    ~employment_history['employee_id']
    .isin(
        people_data['employee_id']
    )
]
len(missing_employees)

0

# SUMMARY

In [38]:
quality_summary = pd.DataFrame({

    "validation": [
        "Null Check",
        "Duplicate Check",
        "Hierarchy Validation",
        "Salary Validation"
    ],

    "status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

# EXPORT

In [39]:
quality_summary.to_csv(
    "../../docs/data_quality_scorecard.csv",
    index=False
)

# LOAD CLEANED STAGING TABLES

In [40]:
# people_data

people_data.to_sql(
    "clean_people_data",
    con=engine,
    if_exists='replace',
    index=False
)

4138

In [41]:
# employment history

employment_history.to_sql(
    "clean_people_employment_history",
    con=engine,
    if_exists='replace',
    index=False
)

4138